# Example 12.4 / 2: Start from an Infeasible Seed with Phase I

Companion notebook for **12.4 Column Generation** in the AMPL lab Documentation, section "Start from an Infeasible Seed with Phase I". It runs the page's example from top to bottom: the model, the amplpy way (data.py, run.py), then the pure-AMPL way (data.dat, model.run). Solver: HiGHS.

## Install Dependencies
💡 TIP: Visit [portal.ampl.com](https://portal.ampl.com/user/ampl/license/list) or [dev.ampl.com](https://dev.ampl.com/ampl/install.html) for AMPL setup on Colab, Docker, Python, Windows, Linux, macOS

In [ ]:
# Install dependencies
%pip install -q amplpy

# Google Colab & Kaggle integration
from amplpy import AMPL, ampl_notebook
ampl = ampl_notebook(
    modules=["highs"],      # modules to install
    license_uuid="default") # license to use

## Start from an Infeasible Seed with Phase I

A restricted master needs feasible seed columns or an explicit Phase I with artificial variables.
If artificials remain positive, pricing must target the Phase-I objective until feasibility is
restored. An infeasible or unbounded pricing problem is algorithmic evidence that requires
diagnosis; it is not equivalent to “no improving column.” Reject duplicate columns by their full
coefficient vector and cost, not by a display name.

In the next example the seed columns cut only small and medium pieces, so no column covers the
large demand. Fixing the artificials at 0 shows the infeasible master; after `unfix`, the
Phase-I objective `PhaseOne` measures the uncovered demand, and its duals price a
column with zero cost in Phase I. The amplpy run sends `fix` and `unfix` with `ampl.eval`,
because fixing every member of an indexed variable is an AMPL statement:

### model.mod
The model, written to a file that both ways of running it read.

In [2]:
%%writefile model.mod
set ITEMS;                                     # item types cut from stock rolls
param demand {ITEMS} integer >= 0;             # pieces ordered per item
param nPatterns integer >= 0;                  # columns in the seed master
set PATTERNS = 1..nPatterns;                   # current pattern pool
param pieces {ITEMS, PATTERNS} integer >= 0;   # pieces of item i in pattern p

var UsePattern {PATTERNS} >= 0;                # rolls cut with each pattern
var Artificial {ITEMS} >= 0;                   # uncovered demand

minimize PhaseOne: sum {i in ITEMS} Artificial[i];  # Phase I: total uncovered demand
subject to MeetDemand {i in ITEMS}:            # artificials keep the master feasible
    sum {p in PATTERNS} pieces[i,p] * UsePattern[p] + Artificial[i] >= demand[i];

Writing model.mod


### amplpy: data.py and run.py

`ampl` is the `AMPL` object created above. Each run starts from a fresh session that has read `model.mod`; then `data.py` sets the data from Python and `run.py` solves and reads the results.

In [3]:
# A fresh session that has read model.mod, as data.py and run.py assume
ampl.reset()
ampl.eval("reset options;")  # reset keeps the options
ampl.read("model.mod")

In [4]:
# data.py
ampl.set["ITEMS"] = ["small", "medium", "large"]
ampl.param["demand"] = {"small": 5, "medium": 4, "large": 3}
ampl.param["nPatterns"] = 2
ampl.param["pieces"] = {
    ("small", 1): 3, ("medium", 1): 0, ("large", 1): 0,
    ("small", 2): 0, ("medium", 2): 2, ("large", 2): 0,
}

In [5]:
# run.py
ampl.option["presolve"] = 0
ampl.eval("fix {i in ITEMS} Artificial[i] := 0;")
ampl.solve(solver="highs", verbose=False)
print("seed master without artificials:", ampl.solve_result)

ampl.eval("unfix Artificial;")
ampl.solve(solver="highs", verbose=False)
price = ampl.con["MeetDemand"].get_values("dual").to_dict()
print("Phase I:", ampl.solve_result, "PhaseOne", ampl.obj["PhaseOne"].value(), "duals", price)
column = {"small": 0, "medium": 0, "large": 2}
print("Phase-I reduced cost of (0,0,2):", 0 - sum(price[i] * column[i] for i in column))

ampl.param["nPatterns"] = 3
ampl.param["pieces"] = {(i, 3): column[i] for i in column}
ampl.solve(solver="highs", verbose=False)
print("after adding it: PhaseOne", ampl.obj["PhaseOne"].value())

seed master without artificials: infeasible
Phase I: solved PhaseOne 3.0 duals {'large': 1, 'medium': 0, 'small': 0}
Phase-I reduced cost of (0,0,2): -2
after adding it: PhaseOne 0.0


### Pure AMPL: data.dat and model.run

The same in pure AMPL, with the same `model.mod`:

In [6]:
%%writefile data.dat
set ITEMS := small medium large;
param demand := small 5  medium 4  large 3;
param nPatterns := 2;
param pieces:  1  2 :=
    small      3  0
    medium     0  2
    large      0  0 ;

Writing data.dat


In [7]:
%%ampl_eval
reset;
reset options;
model model.mod;
data data.dat;

In [8]:
%%ampl_eval
# model.run
option solver highs, solver_msg 0, presolve 0;
fix {i in ITEMS} Artificial[i] := 0;
solve;
display solve_result;
unfix Artificial;
solve;
display PhaseOne, MeetDemand.dual;
let nPatterns := 3;
let pieces['small', 3] := 0;
let pieces['medium', 3] := 0;
let pieces['large', 3] := 2;
solve;
display PhaseOne;


suffix dunbdd OUT;
solve_result = infeasible

PhaseOne = 3

MeetDemand.dual [*] :=
 large  1
medium  0
 small  0
;

PhaseOne = 0

